# Positive-Unlabeled (PU) Learning for Drug Repurposing

## The Problem with Standard Classification

In our drug repurposing task, we have a **label bias problem**:
- **"Success"** = Drug-disease pairs that worked in clinical trials ✓
- **"Failure"** = Drug-disease pairs that were tried but didn't show clear success

But "failure" doesn't mean the drug *can't* work — it might mean:
- The trial was underpowered
- Wrong dosage was tested
- Trial was stopped for business reasons
- Not enough follow-up time

**PU Learning** treats these as **unlabeled** rather than negative, which is more honest.

## How PU Learning Works

1. **Positive examples** (P): Drug-disease pairs that succeeded
2. **Unlabeled examples** (U): Everything else (including our "failures")

The algorithm learns to distinguish positives from the unlabeled set, assuming some unlabeled examples are actually positive.

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, f1_score,
    precision_recall_curve, roc_curve, roc_auc_score,
    average_precision_score, precision_score, recall_score
)

from pulearn import BaggingPuClassifier, ElkanotoPuClassifier

# PU Learning
try:
    from pulearn import BaggingPuClassifier, ElkanotoPuClassifier
    print("✓ pulearn imported successfully")
except ImportError:
    print("Installing pulearn...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'pulearn'])
    from pulearn import BaggingPuClassifier, ElkanotoPuClassifier
    print("✓ pulearn installed and imported")

# XGBoost
try:
    from xgboost import XGBClassifier
    print("✓ xgboost imported successfully")
except ImportError:
    print("XGBoost not available, will use RandomForest only")
    XGBClassifier = None

np.random.seed(42)

# Paths
FIG_PATH = "../results/figures/06-pu-"
TAB_PATH = "../results/tables/06-pu-"
MODEL_PATH = "../results/models/06-pu-"

os.makedirs("../results/figures", exist_ok=True)
os.makedirs("../results/tables", exist_ok=True)
os.makedirs("../results/models", exist_ok=True)

plt.rcParams["figure.figsize"] = (10, 6)

Installing pulearn...


FileNotFoundError: [Errno 2] No such file or directory: 'pip'

## 1. Load Data

In [ ]:
# Load the merged dataset from EDA
with open("../data/03-result/merged_df.pkl", "rb") as f:
    merged_df = pickle.load(f)

with open("../data/02-result/dates_df.pkl", "rb") as f:
    dates_df = pickle.load(f)

# Merge with dates for temporal split
merged_df = merged_df.merge(
    dates_df[["drug_id", "disease_id", "first_trial_date"]],
    how="left",
    on=["drug_id", "disease_id"]
)

print(f"Total samples: {len(merged_df)}")
print(f"Success (P): {merged_df['success'].sum()}")
print(f"Failure (will treat as U): {(~merged_df['success'].astype(bool)).sum()}")

In [ ]:
# Prepare features - drop non-numeric and identifier columns
id_cols = ['drug_id', 'disease_id', 'drug_name', 'disease_name', 'first_trial_date']
path_cols = [c for c in merged_df.columns if 'pathway' in c.lower()]
drop_cols = id_cols + path_cols + ['success']

# Keep only columns that exist
drop_cols = [c for c in drop_cols if c in merged_df.columns]

X = merged_df.drop(columns=drop_cols)
X = X.select_dtypes(include=[np.number])

# Handle missing values
X = X.fillna(X.median())

y = merged_df['success'].astype(int)

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"\nClass distribution:")
print(y.value_counts())

## 2. Temporal Train/Test Split

In [ ]:
# Sort by date and split temporally
merged_df_sorted = merged_df.sort_values('first_trial_date').reset_index(drop=True)
X_sorted = X.loc[merged_df_sorted.index].reset_index(drop=True)
y_sorted = y.loc[merged_df_sorted.index].reset_index(drop=True)

# 85% train, 15% test (temporal)
split_idx = int(len(X_sorted) * 0.85)

X_train = X_sorted.iloc[:split_idx]
X_test = X_sorted.iloc[split_idx:]
y_train = y_sorted.iloc[:split_idx]
y_test = y_sorted.iloc[split_idx:]

# For PU learning: convert labels
# Positive (success=1) stays 1
# Unlabeled (success=0) becomes 0 in PU framework
y_train_pu = y_train.copy()  # Already in correct format for pulearn

print(f"Train: {len(X_train)} samples")
print(f"  - Positive (P): {y_train_pu.sum()}")
print(f"  - Unlabeled (U): {(y_train_pu == 0).sum()}")
print(f"\nTest: {len(X_test)} samples")
print(f"  - Success: {y_test.sum()}")
print(f"  - Failure: {(y_test == 0).sum()}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled!")

## 3. PU Learning with Bagging

The `BaggingPuClassifier` works by:
1. Creating bootstrap samples of the unlabeled data
2. Training multiple classifiers treating each sample as negative
3. Averaging predictions across all classifiers

This helps because some unlabeled examples are actually positive, and bagging helps "average out" this noise.

In [ ]:
# Base classifier for PU learning
base_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# PU Learning classifier
pu_classifier = BaggingPuClassifier(
    estimator=base_classifier,
    n_estimators=15,  # Number of bagging iterations
    n_jobs=-1,
    random_state=42
)

print("Training PU classifier...")
pu_classifier.fit(X_train_scaled, y_train_pu)
print("✓ PU classifier trained!")

In [ ]:
# Also train a standard classifier for comparison
standard_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print("Training standard classifier for comparison...")
standard_classifier.fit(X_train_scaled, y_train)
print("✓ Standard classifier trained!")

## 4. Evaluation & Comparison

In [ ]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    """Evaluate a model and return metrics."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'model': model_name,
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'auc': roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) > 1 else 0,
        'ap': average_precision_score(y_test, y_prob) if len(np.unique(y_test)) > 1 else 0,
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    
    return metrics

# Evaluate both models
pu_metrics = evaluate_model(pu_classifier, X_test_scaled, y_test, "PU Learning")
std_metrics = evaluate_model(standard_classifier, X_test_scaled, y_test, "Standard RF")

# Display comparison
comparison_df = pd.DataFrame([std_metrics, pu_metrics])
comparison_df = comparison_df[['model', 'precision', 'recall', 'f1', 'auc', 'ap']]

print("\n" + "="*60)
print("MODEL COMPARISON ON TEST SET")
print("="*60)
print(comparison_df.to_string(index=False))

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, metrics, title in zip(
    axes, 
    [std_metrics, pu_metrics], 
    ['Standard Classification', 'PU Learning']
):
    cm = confusion_matrix(y_test, metrics['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred Fail', 'Pred Success'],
                yticklabels=['Actual Fail', 'Actual Success'])
    ax.set_title(f"{title}\nF1: {metrics['f1']:.3f}")
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(FIG_PATH + "confusion_matrices.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PR Curve
for metrics, label, color in [
    (std_metrics, 'Standard RF', 'blue'),
    (pu_metrics, 'PU Learning', 'green')
]:
    precision, recall, _ = precision_recall_curve(y_test, metrics['y_prob'])
    axes[0].plot(recall, precision, color=color, 
                 label=f"{label} (AP={metrics['ap']:.3f})")

axes[0].axhline(y=y_test.mean(), color='gray', linestyle='--', label='Baseline')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend()

# ROC Curve
for metrics, label, color in [
    (std_metrics, 'Standard RF', 'blue'),
    (pu_metrics, 'PU Learning', 'green')
]:
    fpr, tpr, _ = roc_curve(y_test, metrics['y_prob'])
    axes[1].plot(fpr, tpr, color=color,
                 label=f"{label} (AUC={metrics['auc']:.3f})")

axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_PATH + "curves_comparison.png", dpi=300)
plt.show()

## 5. PU Learning with XGBoost Base

In [ ]:
if XGBClassifier is not None:
    # XGBoost base classifier
    xgb_base = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=(y_train == 0).sum() / y_train.sum(),
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    
    # PU with XGBoost
    pu_xgb = BaggingPuClassifier(
        estimator=xgb_base,
        n_estimators=10,
        n_jobs=-1,
        random_state=42
    )
    
    print("Training PU-XGBoost...")
    pu_xgb.fit(X_train_scaled, y_train_pu)
    
    pu_xgb_metrics = evaluate_model(pu_xgb, X_test_scaled, y_test, "PU-XGBoost")
    
    print(f"\nPU-XGBoost Results:")
    print(f"  F1: {pu_xgb_metrics['f1']:.4f}")
    print(f"  AUC: {pu_xgb_metrics['auc']:.4f}")
    print(f"  AP: {pu_xgb_metrics['ap']:.4f}")
else:
    print("XGBoost not available")
    pu_xgb = None
    pu_xgb_metrics = None

## 6. Elkanoto PU Classifier

An alternative PU learning approach that estimates the fraction of positives in the unlabeled set.

In [ ]:
# Elkanoto approach - estimates class prior
elkanoto_classifier = ElkanotoPuClassifier(
    estimator=RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),
    hold_out_ratio=0.2
)

print("Training Elkanoto PU classifier...")
elkanoto_classifier.fit(X_train_scaled, y_train_pu)

elkanoto_metrics = evaluate_model(elkanoto_classifier, X_test_scaled, y_test, "Elkanoto PU")

print(f"\nElkanoto PU Results:")
print(f"  F1: {elkanoto_metrics['f1']:.4f}")
print(f"  AUC: {elkanoto_metrics['auc']:.4f}")
print(f"  AP: {elkanoto_metrics['ap']:.4f}")

## 7. Final Comparison

In [ ]:
# Collect all results
all_results = [std_metrics, pu_metrics, elkanoto_metrics]
if pu_xgb_metrics:
    all_results.append(pu_xgb_metrics)

results_df = pd.DataFrame(all_results)
results_df = results_df[['model', 'precision', 'recall', 'f1', 'auc', 'ap']]
results_df = results_df.sort_values('f1', ascending=False)

print("\n" + "="*70)
print("FINAL MODEL COMPARISON")
print("="*70)
print(results_df.to_string(index=False))

# Save results
results_df.to_csv(TAB_PATH + "model_comparison.csv", index=False)
print(f"\nResults saved to {TAB_PATH}model_comparison.csv")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(results_df))
width = 0.25

ax.bar(x - width, results_df['precision'], width, label='Precision', color='steelblue')
ax.bar(x, results_df['recall'], width, label='Recall', color='darkorange')
ax.bar(x + width, results_df['f1'], width, label='F1', color='green')

ax.set_ylabel('Score')
ax.set_title('Model Comparison: Standard vs PU Learning')
ax.set_xticks(x)
ax.set_xticklabels(results_df['model'])
ax.legend()
ax.set_ylim(0, 1)

for i, (p, r, f) in enumerate(zip(results_df['precision'], results_df['recall'], results_df['f1'])):
    ax.text(i - width, p + 0.02, f'{p:.2f}', ha='center', fontsize=8)
    ax.text(i, r + 0.02, f'{r:.2f}', ha='center', fontsize=8)
    ax.text(i + width, f + 0.02, f'{f:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_PATH + "model_comparison_bar.png", dpi=300, bbox_inches='tight')
plt.show()

## 8. Novel Drug Repurposing Predictions with PU Learning

Use the best PU model to predict on untested drug-disease combinations.

In [ ]:
# Select best model
best_model_name = results_df.iloc[0]['model']
print(f"Best model: {best_model_name}")

if 'XGBoost' in best_model_name and pu_xgb is not None:
    best_pu_model = pu_xgb
elif 'Elkanoto' in best_model_name:
    best_pu_model = elkanoto_classifier
elif 'PU' in best_model_name:
    best_pu_model = pu_classifier
else:
    best_pu_model = pu_classifier  # Default to bagging PU

print(f"Using: {type(best_pu_model).__name__}")

In [ ]:
# Get predictions on the full dataset
X_all_scaled = scaler.transform(X.fillna(X.median()))
y_prob_all = best_pu_model.predict_proba(X_all_scaled)[:, 1]

# Add predictions to dataframe
predictions_df = merged_df[['drug_id', 'disease_id', 'drug_name', 'disease_name', 'success']].copy()
predictions_df['pu_probability'] = y_prob_all
predictions_df['pu_prediction'] = (y_prob_all > 0.5).astype(int)

print(f"Predictions generated for {len(predictions_df)} drug-disease pairs")

In [ ]:
# Find "hidden positives" - labeled as failures but predicted as successes
hidden_positives = predictions_df[
    (predictions_df['success'] == 0) & 
    (predictions_df['pu_probability'] > 0.6)  # High confidence
].sort_values('pu_probability', ascending=False)

print("\n" + "="*80)
print("POTENTIAL HIDDEN POSITIVES")
print("These were labeled as 'failures' but PU Learning predicts they might work!")
print("="*80 + "\n")

display_cols = ['drug_name', 'disease_name', 'pu_probability']
print(hidden_positives[display_cols].head(20).to_string(index=False))

In [ ]:
# Distribution of predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram by original label
axes[0].hist(predictions_df[predictions_df['success'] == 1]['pu_probability'], 
             bins=30, alpha=0.7, label='Original Success', color='green')
axes[0].hist(predictions_df[predictions_df['success'] == 0]['pu_probability'], 
             bins=30, alpha=0.7, label='Original Failure', color='red')
axes[0].axvline(x=0.5, color='black', linestyle='--', label='Threshold')
axes[0].set_xlabel('PU Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of PU Predictions by Original Label')
axes[0].legend()

# Box plot
predictions_df.boxplot(column='pu_probability', by='success', ax=axes[1])
axes[1].set_xlabel('Original Label (0=Fail, 1=Success)')
axes[1].set_ylabel('PU Predicted Probability')
axes[1].set_title('PU Predictions by Original Label')
plt.suptitle('')

plt.tight_layout()
plt.savefig(FIG_PATH + "prediction_distribution.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Save predictions
predictions_df.to_csv("../results/pu_learning_predictions.csv", index=False)
hidden_positives.to_csv("../results/pu_hidden_positives.csv", index=False)

print(f"\n✓ Saved {len(predictions_df)} predictions to ../results/pu_learning_predictions.csv")
print(f"✓ Saved {len(hidden_positives)} hidden positives to ../results/pu_hidden_positives.csv")

In [ ]:
# Save best model
with open(MODEL_PATH + "best_pu_model.pkl", "wb") as f:
    pickle.dump(best_pu_model, f)

with open(MODEL_PATH + "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print(f"\n✓ Model saved to {MODEL_PATH}best_pu_model.pkl")

## 9. Summary

### What PU Learning Does Differently

| Standard Classification | PU Learning |
|-------------------------|-------------|
| "Failure" = Definitely won't work | "Failure" = Unknown, might work |
| Learns hard boundary | Learns soft probabilities |
| Conservative on positives | More willing to predict positives |
| May miss hidden opportunities | Finds "hidden positives" |

### Key Outputs

1. **`pu_learning_predictions.csv`** - All drug-disease pairs with PU probabilities
2. **`pu_hidden_positives.csv`** - "Failures" that PU thinks might actually work
3. **`best_pu_model.pkl`** - Saved model for future predictions

### Interpretation

The "hidden positives" are the most interesting output — these are drug-disease combinations that:
- Were tried in clinical trials
- Were labeled as "failure" (no clear success)
- But PU Learning thinks they might actually work

These could be candidates for:
- Re-analysis of original trial data
- New trials with better design (dosage, patient selection)
- Literature review to find supporting evidence